In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

user_id,first_name,last_name,email,signup_date,country,referral_source
USR_1000,Linda,Smith,linda.smith0@example.com,2023-08-01T00:00:00.000Z,UK,google_ads
USR_1001,David,White,david.white1@example.com,2022-08-12T00:00:00.000Z,AU,organic
USR_1002,Grace,Wilson,grace.wilson2@example.com,2022-03-21T00:00:00.000Z,US,organic
USR_1003,Susan,Davis,susan.davis3@example.com,2024-11-14T00:00:00.000Z,AU,organic
USR_1004,Emma,Miller,emma.miller4@example.com,2024-05-23T00:00:00.000Z,UK,social_media
USR_1005,Grace,Rodriguez,grace.rodriguez5@example.com,2022-01-28T00:00:00.000Z,UK,social_media
USR_1006,Lisa,Rodriguez,lisa.rodriguez6@example.com,2022-11-29T00:00:00.000Z,UK,referral
USR_1007,William,Williams,william.williams7@example.com,2024-03-03T00:00:00.000Z,US,referral
USR_1008,Daniel,Martin,daniel.martin8@example.com,2023-07-10T00:00:00.000Z,US,social_media
USR_1009,Edward,Jones,edward.jones9@example.com,2024-02-29T00:00:00.000Z,US,partner


In [0]:
@dp.materialized_view(
    name="gold_daily_signups_v2",
    comment="Analysis of user signups by day of week"
)
def gold_daily_signups_v2():
    df_silver = spark.read.table("clean_data_silver_v2")
    
    return df_silver.withColumn('day_of_week', F.dayofweek('signup_date')) \
        .withColumn('day_name', F.date_format('signup_date', 'EEEE')) \
        .groupBy('day_of_week', 'day_name') \
        .agg(
            F.count('user_id').alias('total_signups')
        ) \
        .orderBy(F.desc('total_signups'))

day_of_week,day_name,total_signups
5,Thursday,14
3,Tuesday,9
2,Monday,7
1,Sunday,7
7,Saturday,5
6,Friday,4
4,Wednesday,4


In [0]:
@dp.materialized_view(
    name="gold_referral_analysis_v2",
    comment="Analysis of which referral sources drive the most signups and their geographic reach"
)
def gold_referral_analysis_v2():
    df_silver = spark.read.table("clean_data_silver_v2")
    
    return df_silver.groupBy('referral_source') \
        .agg(
            F.count('user_id').alias('total_clicks'),
            F.countDistinct('country').alias('countries_reached')
        ) \
        .orderBy(F.desc('total_clicks'))

referral_source,total_clicks,countries_reached
social_media,14,5
organic,11,5
referral,9,4
google_ads,8,5
partner,8,3


In [0]:
@dp.materialized_view(
    name="insights_gold_v2",
    comment="Comprehensive insights combining day of week and referral source analysis"
)
def insights_gold_v2():
    df_silver = spark.read.table("clean_data_silver_v2")
    
    return df_silver.withColumn('day_of_week', F.dayofweek('signup_date')) \
        .withColumn('day_name', F.date_format('signup_date', 'EEEE')) \
        .groupBy('day_name', 'referral_source') \
        .agg(
            F.count('user_id').alias('signups'),
            F.countDistinct('country').alias('unique_countries')
        ) \
        .orderBy(F.desc('signups'))

day_name,referral_source,signups,unique_countries
Thursday,referral,4,3
Thursday,organic,3,3
Tuesday,referral,3,3
Thursday,social_media,3,2
Monday,social_media,3,3
Saturday,social_media,3,3
Wednesday,social_media,2,2
Sunday,organic,2,2
Monday,organic,2,2
Thursday,google_ads,2,2


In [0]:
# This cell is no longer needed.
# The @dp.materialized_view decorators handle table creation automatically.
# No need for .write.saveAsTable() operations in Spark Declarative Pipelines.

In [0]:
# This cell is no longer needed.
# The pipeline automatically manages all table creation and updates.
# To preview data, query the tables after running the pipeline.

day_of_week,day_name,total_signups
5,Thursday,14
3,Tuesday,9
2,Monday,7
1,Sunday,7
7,Saturday,5
4,Wednesday,4
6,Friday,4
